<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-04/AI_Modul_4.1_Praktikum_Pengantar_Data_Science.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 4.1: Praktikum Pengantar Data Science
## Penerapan Metodologi CRISP-DM pada Analitik Pertanian Presisi Kelapa Sawit

Notebook ini mengimplementasikan alur kerja data science terstruktur (*end-to-end*) berdasarkan kerangka **CRISP-DM** (*Cross-Industry Standard Process for Data Mining*) untuk estimasi produktivitas Tandan Buah Segar (TBS) kelapa sawit.

### Agenda Praktikum:
1. **Verifikasi Lingkungan Komputasi & Import Pustaka**
2. **CRISP-DM 1 & 2: Sintesis & Pemahaman Data Operasional Kebun**
3. **CRISP-DM 3: Pembersihan Data, Deteksi Anomali, dan Rekayasa Fitur**
4. **CRISP-DM 4: Pemodelan Regresi Baseline Pertanian Presisi**
5. **CRISP-DM 5: Evaluasi Metrik Kinerja (R², MAE, RMSE)**
6. **CRISP-DM 6: Penyiapan Logika Deployment untuk Sistem Logistik Pabrik**

In [ ]:
# 1. Inisialisasi Environment & Import Pustaka
import os
os.environ['MPLBACKEND'] = 'Agg'  # Backend non-interaktif untuk eksekusi headless

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print(f"Versi NumPy       : {np.__version__}")
print(f"Versi Pandas      : {pd.__version__}")
print(f"Versi Seaborn     : {sns.__version__}")
print("Lingkungan Data Science siap digunakan!")

## 2. CRISP-DM Fase 1 & 2: Business & Data Understanding

**Tujuan Bisnis:** Memprediksi hasil panen TBS (Ton/Hektar) per blok pertanaman untuk mengoptimalkan logistik truk angkut ke PKS.
**Akuisisi Data:** Memuat 250 blok observasi kebun dengan data telemetri cuaca, umur tanaman, dan indeks kanopi drone (NDVI).

In [ ]:
# Membuat Dataset Sintetis Realistis Perkebunan Sawit (250 Blok)
np.random.seed(42)
n_samples = 250

blok_id = [f"BLK-{i+1:03d}" for i in range(n_samples)]
afdeling = np.random.choice(["AFD-I", "AFD-II", "AFD-III", "AFD-IV"], size=n_samples)
umur_tanaman = np.random.randint(5, 26, size=n_samples) # Usia 5-25 tahun
curah_hujan = np.random.uniform(140.0, 340.0, size=n_samples) # mm/bulan
hari_hujan = np.random.randint(8, 22, size=n_samples)
pupuk_npk = np.random.uniform(2.0, 4.5, size=n_samples) # kg/pokok
indeks_ndvi = np.random.uniform(0.55, 0.88, size=n_samples) # Indeks vegetasi kanopi
kelembaban_tanah = np.random.uniform(24.0, 42.0, size=n_samples) # % lengas tanah

# Formulasi biologis target panen (Ton/Ha) dengan noise lingkungan
# Kurva usia tanaman sawit: produktivitas puncak pada usia 9-16 tahun
faktor_usia = -0.04 * (umur_tanaman - 13)**2 + 22.0
faktor_iklim = 0.025 * curah_hujan + 5.5 * indeks_ndvi + 1.2 * pupuk_npk
noise = np.random.normal(0, 1.2, size=n_samples)
tonase_tbs = 0.45 * faktor_usia + 0.35 * faktor_iklim + noise
tonase_tbs = np.round(np.clip(tonase_tbs, 8.5, 32.0), 2)

df_sawit = pd.DataFrame({
    'Blok_ID': blok_id,
    'Afdeling': afdeling,
    'Umur_Tanaman_Thn': umur_tanaman,
    'Curah_Hujan_mm': np.round(curah_hujan, 1),
    'Hari_Hujan': hari_hujan,
    'Dosis_NPK_kg': np.round(pupuk_npk, 2),
    'Indeks_NDVI': np.round(indeks_ndvi, 3),
    'Kelembaban_Tanah_pct': np.round(kelembaban_tanah, 1),
    'Tonase_TBS_Ton_Ha': tonase_tbs
})

# Suntikkan 3 missing value dan 2 outlier sensorik untuk studi kasus pembersihan
df_sawit.loc[12, 'Curah_Hujan_mm'] = np.nan
df_sawit.loc[45, 'Indeks_NDVI'] = np.nan
df_sawit.loc[88, 'Kelembaban_Tanah_pct'] = np.nan
df_sawit.loc[70, 'Kelembaban_Tanah_pct'] = 98.5 # Sensor terendam genangan air ekstrem

print("Tampilan 5 Baris Pertama Dataset Kebun Sawit:")
df_sawit.head()

In [ ]:
# Profiling Data: Struktur, Tipe Data, dan Nilai Hilang
print("Informasi Ringkas Metadata Kolom:")
print(df_sawit.info())
print("\nJumlah Nilai Hilang (Missing Values) Per Kolom:")
print(df_sawit.isnull().sum())

## 3. CRISP-DM Fase 3: Data Preparation (Pembersihan & Rekayasa Fitur)

Pada tahap ini kita menyelesaikan anomali data:
1. **Imputasi Nilai Hilang:** Menggunakan median per kelompok atau median global agar tahan (*robust*) terhadap skewness.
2. **Penanganan Outlier:** Mendeteksi nilai tidak masuk akal secara fisik (kelembaban tanah > 60% pada tanah mineral dikoreksi dengan metode batas atas kuartil IQR).
3. **Rekayasa Fitur (*Feature Engineering*):** Menghitung rasio curah hujan per hari hujan sebagai intensitas hujan harian.

In [ ]:
# 1. Imputasi Nilai Hilang
df_clean = df_sawit.copy()
df_clean['Curah_Hujan_mm'] = df_clean['Curah_Hujan_mm'].fillna(df_clean['Curah_Hujan_mm'].median())
df_clean['Indeks_NDVI'] = df_clean['Indeks_NDVI'].fillna(df_clean['Indeks_NDVI'].median())
df_clean['Kelembaban_Tanah_pct'] = df_clean['Kelembaban_Tanah_pct'].fillna(df_clean['Kelembaban_Tanah_pct'].median())

# 2. Deteksi & Penanganan Outlier Sensor (Metode IQR)
Q1 = df_clean['Kelembaban_Tanah_pct'].quantile(0.25)
Q3 = df_clean['Kelembaban_Tanah_pct'].quantile(0.75)
IQR = Q3 - Q1
batas_atas = Q3 + 1.5 * IQR
print(f"Batas Atas Kelembaban Tanah: {batas_atas:.2f}%")

# Kliping nilai kelembaban tanah ke batas atas
df_clean['Kelembaban_Tanah_pct'] = np.where(
    df_clean['Kelembaban_Tanah_pct'] > batas_atas,
    batas_atas,
    df_clean['Kelembaban_Tanah_pct']
)

# 3. Rekayasa Fitur: Intensitas Hujan Harian
df_clean['Intensitas_Hujan_mm_hari'] = np.round(df_clean['Curah_Hujan_mm'] / df_clean['Hari_Hujan'], 2)

print("Pemeriksaan Nilai Hilang Pasca-Pembersihan:")
print(df_clean.isnull().sum())
print("\nDeskriptif Statistik Fitur Numerik:")
df_clean.describe()

### Eksplorasi Visual: Matriks Korelasi Pearson

In [ ]:
# Visualisasi Korelasi Antar-Fitur Terhadap Tonase Panen TBS
plt.figure(figsize=(9, 6), dpi=150)
fitur_numerik = ['Umur_Tanaman_Thn', 'Curah_Hujan_mm', 'Hari_Hujan', 
                 'Dosis_NPK_kg', 'Indeks_NDVI', 'Kelembaban_Tanah_pct', 
                 'Intensitas_Hujan_mm_hari', 'Tonase_TBS_Ton_Ha']
corr_matrix = df_clean[fitur_numerik].corr()

sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='YlGnBu', cbar=True, square=True)
plt.title('Matriks Korelasi Pearson Fitur Agronomi Terhadap Produksi TBS', fontsize=12, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('e:/Project Buku/docs/assets/praktikum_4_1_korelasi.png', dpi=150)
print("Plot korelasi berhasil disimpan ke docs/assets/praktikum_4_1_korelasi.png")

## 4. CRISP-DM Fase 4 & 5: Modeling & Evaluation

Kita membagi data menjadi set latih (*training set* 80%) dan set uji (*testing set* 20%), lalu melatih model regresi linier sebagai model dasar (*baseline*).

In [ ]:
# Menyiapkan Matriks Fitur X dan Vektor Target y
X = df_clean[['Umur_Tanaman_Thn', 'Curah_Hujan_mm', 'Dosis_NPK_kg', 'Indeks_NDVI', 'Kelembaban_Tanah_pct']]
y = df_clean['Tonase_TBS_Ton_Ha']

# Pembagian Dataset (Train 80%, Test 20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Ukuran Data Latih : {X_train.shape[0]} blok")
print(f"Ukuran Data Uji   : {X_test.shape[0]} blok")

# Pelatihan Model Regresi Linier
model = LinearRegression()
model.fit(X_train, y_train)

# Prediksi pada Data Uji
y_pred = model.predict(X_test)

# Evaluasi Metrik Kinerja
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("\n--- Hasil Evaluasi Kinerja Model ---")
print(f"Koefisien Determinasi (R2) : {r2:.4f}")
print(f"Mean Absolute Error (MAE)   : {mae:.4f} Ton/Ha")
print(f"Root Mean Sq Error (RMSE)   : {rmse:.4f} Ton/Ha")

# Menampilkan Bobot Koefisien Setiap Fitur
df_koef = pd.DataFrame({
    'Fitur': X.columns,
    'Koefisien': model.coef_
}).sort_values(by='Koefisien', ascending=False)
print("\nBobot Pengaruh Fitur terhadap Estimasi Produksi TBS:")
print(df_koef.to_string(index=False))

In [ ]:
# Visualisasi Hasil Prediksi vs Aktual
plt.figure(figsize=(8, 6), dpi=150)
plt.scatter(y_test, y_pred, color='#2563EB', alpha=0.7, edgecolors='k', label='Blok Data Uji')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Garis Ideal (y = y_pred)')
plt.xlabel('Nilai Aktual Produksi TBS (Ton/Ha)', fontsize=11, fontweight='bold')
plt.ylabel('Nilai Prediksi Model (Ton/Ha)', fontsize=11, fontweight='bold')
plt.title(f'Evaluasi Model Prediksi TBS Sawit (R2 = {r2:.2f})', fontsize=12, fontweight='bold', pad=12)
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.savefig('e:/Project Buku/docs/assets/praktikum_4_1_evaluasi.png', dpi=150)
print("Plot evaluasi berhasil disimpan ke docs/assets/praktikum_4_1_evaluasi.png")

## 5. CRISP-DM Fase 6: Deployment & Logika Operasional Kebun

Model yang telah divalidasi dikemas ke dalam fungsi penentu kebutuhan armada transportasi panen (*truck dispatching logic*) untuk mencegah buah menginap lebih dari 24 jam di TPH.

In [ ]:
def alokasi_truk_panen(nama_blok, luas_ha, umur, curah_hujan, pupuk, ndvi, kelembaban, kapasitas_truk_ton=7.0):
    """
    Menghitung estimasi total tonase dan kebutuhan armada truk pengangkut TBS ke PKS.
    """
    fitur_input = pd.DataFrame([[umur, curah_hujan, pupuk, ndvi, kelembaban]], columns=X.columns)
    estimasi_ton_per_ha = model.predict(fitur_input)[0]
    total_produksi_ton = estimasi_ton_per_ha * (luas_ha / 12.0) # Estimasi rotasi 1 putaran panen bulanan
    jumlah_truk_dibutuhkan = int(np.ceil(total_produksi_ton / kapasitas_truk_ton))
    
    status_logistik = "NORMAL" if jumlah_truk_dibutuhkan <= 3 else "PRIORITAS TINGGI (LONJAKAN PANEN)"
    
    return {
        "Blok": nama_blok,
        "Luas (Ha)": luas_ha,
        "Estimasi Yield (Ton/Ha)": round(estimasi_ton_per_ha, 2),
        "Total Panen Rotasi (Ton)": round(total_produksi_ton, 2),
        "Kebutuhan Truk": jumlah_truk_dibutuhkan,
        "Status Logistik": status_logistik
    }

# Contoh Simulasi Penugasan Operasional Kebun
simulasi_blok_1 = alokasi_truk_panen("BLK-B04", luas_ha=25.0, umur=12, curah_hujan=260.0, pupuk=3.5, ndvi=0.82, kelembaban=32.0)
simulasi_blok_2 = alokasi_truk_panen("BLK-D11", luas_ha=30.0, umur=22, curah_hujan=160.0, pupuk=2.2, ndvi=0.62, kelembaban=26.0)

df_rekomendasi = pd.DataFrame([simulasi_blok_1, simulasi_blok_2])
print("Rekomendasi Operasional Armada Logistik TBS ke PKS:")
print(df_rekomendasi.to_string(index=False))
print("\nPraktikum Modul 4.1 Selesai dengan Sukses!")